In [2]:
import pandas as pd
import numpy as np
import os
from typing import List, Tuple

# Scikit-learn for the baseline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# --- Define Base Path ---
BASE_DATA_PATH = './data'

In [3]:
pip install numpy 

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
def parse_conllu_text(file_path: str) -> List[str]:
    sentences = []
    current_sentence_tokens = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                if current_sentence_tokens:
                    sentences.append(" ".join(current_sentence_tokens))
                    current_sentence_tokens = []
                continue
            if line.startswith("#"):
                continue
            parts = line.split('\t')
            if len(parts) > 1 and parts[0].isdigit():
                token = parts[1]
                current_sentence_tokens.append(token)
    if current_sentence_tokens:
        sentences.append(" ".join(current_sentence_tokens))
    return sentences

In [5]:
# --- Data Loading (Unlabeled) ---
def load_unlabeled_data(language: str, n_samples: int = 0) -> List[str]:
    """Loads clean text sentences from the CoNLL-U file."""
    conllu_path = os.path.join(BASE_DATA_PATH, f'output_{language}.conllu')
    
    if not os.path.exists(conllu_path):
        print(f"File not found: {conllu_path}. Skipping language.")
        return []
        
    print(f"Loading text data from: {conllu_path}")
    cleaned_texts = parse_conllu_text(conllu_path)
    
    if n_samples > 0 and len(cleaned_texts) > n_samples:
        cleaned_texts = cleaned_texts[:n_samples]
    
    print(f"Data loaded: {len(cleaned_texts)} samples.")
    return cleaned_texts

In [6]:
def run_lda_baseline(texts: List[str], language: str, n_topics: int = 10, n_top_words: int = 10):
    """
    Performs LDA Topic Modeling and evaluates qualitatively and quantitatively.
    """
    print(f"\n--- Baseline: LDA Topic Modeling ({language.upper()}) ---")
    
    # 1. Feature Extraction: Count Vectorizer
    # LDA uses raw term frequency, not TF-IDF.
    # Max_features limits vocabulary size for performance.
    vectorizer = CountVectorizer(
        max_df=0.90, 
        min_df=5, 
        max_features=10000, 
        stop_words=None # No default stop words, let min_df and max_df handle frequent/rare words
    )
    X_counts = vectorizer.fit_transform(texts)
    
    # 2. Model Training: LDA
    lda = LatentDirichletAllocation(
        n_components=n_topics, 
        random_state=42, 
        learning_method='batch', # Use 'batch' for better results on moderate size
        max_iter=5
    )
    print(f"Training LDA with {n_topics} topics...")
    lda.fit(X_counts)
    
    # 3. Quantitative Evaluation: Perplexity
    # Perplexity measures how well the model predicts the sample. Lower is better.
    perplexity = lda.perplexity(X_counts)
    print("\n--- Quantitative Evaluation ---")
    print(f"Perplexity (Lower is Better): {perplexity:.2f}")

    # 4. Qualitative Evaluation: Displaying Topics
    print("\n--- Qualitative Evaluation: Top Words per Topic ---")
    feature_names = vectorizer.get_feature_names_out()
    
    for topic_idx, topic in enumerate(lda.components_):
        top_words_indices = topic.argsort()[:-n_top_words - 1:-1]
        top_words = [feature_names[i] for i in top_words_indices]
        print(f"Topic #{topic_idx + 1}: {' | '.join(top_words)}")

In [7]:
# List of all languages you processed
LANGUAGES = ['be', 'de', 'en', 'es', 'fr', 'it', 'ko', 'pt', 'ru', 'ta']
# Adjust sample size based on your system's memory. 0 means use all available.
# Start with a large number (e.g., 500,000) if you have ample RAM.
SAMPLE_SIZE = 100000 
N_TOPICS = 10

for lang in LANGUAGES:
    print(f"\n=======================================================")
    print(f"STARTING LDA BASELINE FOR LANGUAGE: {lang.upper()}")
    print(f"=======================================================")
    
    texts = load_unlabeled_data(language=lang, n_samples=SAMPLE_SIZE)
    
    if texts:
        run_lda_baseline(texts, lang, n_topics=N_TOPICS)
    else:
        print(f"Skipped {lang.upper()} - no data loaded.")


STARTING LDA BASELINE FOR LANGUAGE: BE
Loading text data from: ./data\output_be.conllu
Data loaded: 100000 samples.

--- Baseline: LDA Topic Modeling (BE) ---
Training LDA with 10 topics...

--- Quantitative Evaluation ---
Perplexity (Lower is Better): 3262.34

--- Qualitative Evaluation: Top Words per Topic ---
Topic #1: на | па | не | фк | што | ігракі | свету | футболе | трэнеры | зборнай
Topic #2: за | вобласці | года | цэнтр | горад | па | гісторыя | каля | тыс | горада
Topic #3: год | дзень | стагоддзя | па | года | га | гадоў | нашай | эры | тысячагоддзя
Topic #4: на | аб | да | ад | пад | горад | тэрыторыі | годзе | таксама | краіны
Topic #5: года | на | раёна | сельсавет | чалавек | да | сельсаветы | скасаваны | насельніцтва | 2006
Topic #6: года | годзе | км | беларускі | на | да | 12 | дзеяч | каля | быў
Topic #7: на | годзе | ст | спасылкі | па | літаратура | раёны | царква | беларускай | быў
Topic #8: да | на | для | ад | па | што | якія | як | насельніцтва | таксама
Topi